In [ ]:
!pip install -q ultralytics gdown

In [ ]:
import ultralytics
ultralytics.checks()

In [ ]:
!curl https://rclone.org/install.sh | sudo bash
!rclone version

In [ ]:
!mkdir -p /root/.config/rclone

In [ ]:
%%writefile /root/.config/rclone/rclone.conf
[gdrive]
type = drive
scope = drive
token = 
team_drive =

In [ ]:
VISIO_ZIP_LINK = "https://drive.google.com/file/d/1TspZqVdesLP0b6884W82ACC0vreyQPj6/view?usp=drive_link"

In [ ]:
!gdown --fuzzy "{VISIO_ZIP_LINK}" -O /kaggle/working/VisioDECT.zip
!ls -lh /kaggle/working/VisioDECT.zip

In [ ]:
!unzip -q /kaggle/working/VisioDECT.zip -d /kaggle/working/visio_raw

In [ ]:
!find /kaggle/working/visio_raw -maxdepth 4 -type d | head -80

In [ ]:
VISIO_PATH = "/kaggle/working/visio_raw/VisioDECT_YOLO_GroupSplit"

yaml_text = f"""
path: {VISIO_PATH}
train: images/train
val: images/val
test: images/test

names:
  0: drone
"""

with open("/kaggle/working/visiodect.yaml", "w") as f:
    f.write(yaml_text)

print(open("/kaggle/working/visiodect.yaml").read())

In [ ]:
!yolo detect train \
  model=yolov8m.pt \
  data=/kaggle/working/visiodect.yaml \
  imgsz=640 \
  epochs=50 \
  batch=64 \
  seed=42 \
  patience=10 \
  save_period=5 \
  device=0,1 \
  project=/kaggle/working/yolo_runs \
  name=yolov8m_visiodect_img640_50ep_b64

In [ ]:
!rclone copy \
  /kaggle/working/yolo_runs/yolov8m_visiodect_img640_50ep_b64 \
  gdrive:yolov8_VisioDECT/yolov8m_visiodect_img640_50ep_b64 \
  --drive-root-folder-id 12UKaDVvGAFa53hrADJdyQup_k6lDkWt0 \
  --progress

In [ ]:
!yolo detect val \
  model=/kaggle/working/yolo_runs/yolov8m_visiodect_img640_50ep_b64/weights/best.pt \
  data=/kaggle/working/visiodect.yaml \
  imgsz=640 \
  split=test \
  device=0 \
  project=/kaggle/working/yolo_runs \
  name=visiodect_yolov8m_test_eval

In [ ]:
!rclone copy \
  /kaggle/working/yolo_runs/visiodect_yolov8m_test_eval \
  gdrive:yolov8_VisioDECT/visiodect_yolov8m_test_eval \
  --drive-root-folder-id 12UKaDVvGAFa53hrADJdyQup_k6lDkWt0 \
  --progress

In [ ]:
UAV300_ZIP_LINK = "https://drive.google.com/file/d/1YfzPoT1ki1K5o4jVEOL0Op1yLddxmc0F/view?usp=drive_link"

!gdown --fuzzy "{UAV300_ZIP_LINK}" -O /kaggle/working/UAV300_YOLO.zip
!ls -lh /kaggle/working/UAV300_YOLO.zip

In [ ]:
!unzip -q /kaggle/working/UAV300_YOLO.zip -d /kaggle/working/uav300_raw

In [ ]:
UAV300_PATH = "/kaggle/working/uav300_raw/Anti-UAV-RGB-YOLO_fit"

In [ ]:
yaml_text = f"""
path: {UAV300_PATH}
train: images/train
val: images/val
test: images/test

names:
  0: drone
"""

with open("/kaggle/working/uav300.yaml", "w") as f:
    f.write(yaml_text)

print(open("/kaggle/working/uav300.yaml").read())

In [ ]:
VISIO_MODEL = "/kaggle/working/yolo_runs/yolov8m_visiodect_img640_50ep_b64/weights/best.pt"

!yolo detect val \
  model="{VISIO_MODEL}" \
  data=/kaggle/working/uav300.yaml \
  imgsz=640 \
  split=test \
  device=0 \
  project=/kaggle/working/yolo_runs \
  name=cross_eval_train_visiodect_test_uav300

In [ ]:
!find "{UAV300_PATH}/images/test" -type f | head -200 > /kaggle/working/uav300_test_sample.txt

In [ ]:
!yolo detect predict \
  model="{VISIO_MODEL}" \
  source=/kaggle/working/uav300_test_sample.txt \
  imgsz=640 \
  conf=0.25 \
  device=0 \
  save=True \
  project=/kaggle/working/yolo_runs \
  name=cross_pred_train_visiodect_test_uav300_sample

In [ ]:
!zip -r /kaggle/working/cross_test_visiodect_to_uav300_results.zip \
  /kaggle/working/yolo_runs/cross_eval_train_visiodect_test_uav300 \
  /kaggle/working/yolo_runs/cross_pred_train_visiodect_test_uav300_sample

In [ ]:
!ls -lh /kaggle/working/cross_test_visiodect_to_uav300_results.zip

In [ ]:
!rclone copy \
  /kaggle/working/cross_test_visiodect_to_uav300_results.zip \
  gdrive:yolov8_VisioDECT/cross_tests \
  --drive-root-folder-id 12UKaDVvGAFa53hrADJdyQup_k6lDkWt0 \
  --progress

In [ ]:
!rclone copy \
  /kaggle/working/yolo_runs/cross_eval_train_visiodect_test_uav300 \
  gdrive:yolov8_VisioDECT/cross_tests/cross_eval_train_visiodect_test_uav300 \
  --drive-root-folder-id 12UKaDVvGAFa53hrADJdyQup_k6lDkWt0 \
  --progress

In [ ]:
!rclone copy \
  /kaggle/working/yolo_runs/cross_pred_train_visiodect_test_uav300_sample \
  gdrive:yolov8_VisioDECT/cross_tests/cross_pred_train_visiodect_test_uav300_sample \
  --drive-root-folder-id 12UKaDVvGAFa53hrADJdyQup_k6lDkWt0 \
  --progress